## Sampling the datasets

The first dataset used in the paper comprises Quora questions. It was made for training a model to classify whether two questions hold equivalent meaning. Due to the nature of the task that the dataset was made for, the same sentence, as identified by the corresponding `qid` feature, can repeat across multiple pairs of questions (text fields `question1` and `question2`), with different `is_duplicate` values. An example would be if `qid=1` is a duplicate of `qid=2` but not of `qid=3`, and pairs `(1,2)` and `(1,3)` both appear in the dataset; `qid=1` appears twice. Since we are not interested in duplicates, we find that the best approach is to extract all distinct sentences by `qid` and then sample that set for ~164k sentences, matching the size of the dataset used by the original paper's authors.

In [1]:
import pandas as pd
from datasets import load_dataset

qqp = load_dataset('AlekseyKorshuk/quora-question-pairs')
df = qqp['train'].to_pandas()

# two new dataframes, projecting over the two (qid, question) pairs
df_q1 = df[['qid1', 'question1']]
df_q2 = df[['qid2', 'question2']]

# join them into a (qid, question) dataframe with duplicate qids
df_dup = pd.concat(
    [df_q1.rename(columns={'qid1': 'qid', 'question1': 'question'}),
     df_q2.rename(columns={'qid2': 'qid', 'question2': 'question'})]
)

# remove duplicate entries
# dataset goes from ~809k entries to ~538k
df_dedup = df_dup.drop_duplicates(subset=['qid']).dropna(how='any', axis=0)

# sample, we used a fixed seed=1
qqp_sample = df_dedup.sample(n=164246, random_state=1).reset_index(drop=True)

# make df to save to disk
pd.DataFrame({
    'id': range(len(qqp_sample)),
    'src': 'qqp',
    'txt': qqp_sample['question'].astype(str)
}).to_parquet('../datasets/qqp.parquet', index=False)

/home/mihnea/Desktop/blackbox-re26/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


The next dataset is QNLI, based on Wikipedia questions and answers. Here the dataset presents as a pair of questions and text responses in features `text1` and `text2`, respectively, with an additional feature `label_text` indicating `entailment` or `not entailment` based on whether or not the response is an answer to the question. The questions and answers both include duplicates. After removing duplicates and null values from among the `text2` values, we obtain exactly 58644 text entries, which is off-by-one to the number obtained by the original paper's authors. Without removing null values with `dropna()`, it is exactly the paper's 58645 entries.

In [ ]:
qnli = load_dataset('SetFit/qnli')
df = qnli['train'].to_pandas()

# project on the text2 field
df_proj = df[['text2']]

# drop duplicate sentences and nulls
# this df has exactly 58644 entries, corresponding w/ the paper
df_dedup = df_proj.drop_duplicates().dropna(how='any', axis=0)

# save to disk
pd.DataFrame({
    'id': range(len(df_dedup)),
    'src': 'qnli',
    'txt': df_dedup['text2'].astype(str)
}).to_parquet('../datasets/qnli.parquet', index=False)

Repo card metadata block was not found. Setting CardData to empty.


,text1
0,When did the third Digimon series begin?
1,Which missile batteries often have individual ...
2,What two things does Popper argue Tarski's the...
3,What is the name of the village 9 miles north ...
4,What famous palace is located in London?
...,...
104738,What is Fairfield Court?
104739,Who did William the Conqueror give the site of...
104740,What percentage of high-density penetrators is...
104741,What individual was responsible for law and ma...


The next dataset is the English Wikipedia dataset. Bolukbasi et al. indicate that their method of preparing the dataset is the same as that used by Devlin et al. in their 2018 paper on BERT. However, the Devlin et al. paper is unclear as to the method/algorithm it uses to preprocess data from the `wikimedia/wikipedia` dataset, even though it mentions to clean out filler (specifically, to "ignore lists, tables, and headers"). This task is not trivial as the `wikimedia/wikipedia` dataset is a raw dump with no cleaning performed. 

We have attempted to employ various strategies to filter and clean this dataset ourselves, but our results produced unsatisfactory data. In the absence of a definite method, we have opted to use a different English Wikipedia dataset (namely `google/wiki40b`) pre-cleaned by its maintainers (Google), which lends itself easily to paragraph extraction. We then use a `spacy` model `en_core_web_sm` to split these paragraphs into sentences, saving 203736 of them as our dataset. While our approach has to be different for the reason stated above, this dataset still preserves the core idea of Devlin et al., namely to "use a document-level corpus rather than a shuffled sentence-level corpus"; as such, we believe this dataset will produce qualitatively similar results, in that the dataset-level concepts identified by Bolukbasi et al. still emerge.

In [21]:
import spacy
import numpy as np

wiki = load_dataset('google/wiki40b', 'en', split='train')

rng = np.random.default_rng(1)
# this is more articles than we'll actually need
# we'll process all of them and stop adding after 203736
idx = rng.choice(len(wiki), size=15000, replace=False)

# turn them into a df
articles = wiki.select(idx)

In [14]:
# model to split paragraphs into sentences
model = spacy.load('en_core_web_sm')
# collect sentences produced by it here
sents = []

for article in articles:
    # take the article text
    text = article['text']
    # and extract all the paragraphs from it
    # google/wiki40b is formatted for this purpose
    ps = []
    for item in text.split('_START_'):
        if item.startswith('PARAGRAPH_'):
            # remove the string 'PARAGRAPH_'
            p = item[10:]
            # and any newline markers present
            p = p.replace('_NEWLINE_', '')
            # collect the paragraph
            ps.append(p.strip())

    # let the model parse each paragraph
    for doc in model.pipe(ps):
        # it produces sentences to add
        for sent in doc.sents:
            sents.append(sent)

In [23]:
# 328515 sentences, in excess of the target
df = pd.Series(sents)
# this doesn't drop anything here, but is still a good idea
# e.g. if the seed is changed and on that random draw there are duplicates
df_dedup = df.drop_duplicates()

# take only what we need
# the last article might be truncated this way
# this is acceptable, since the rest is OK
df_head = df_dedup.head(203736)

# and save to disk
pd.DataFrame({
    'id': range(len(df_head)),
    'src': 'wiki',
    'txt': df_head.astype(str).to_numpy()
}).to_parquet('../datasets/wiki.parquet', index=False)

The final dataset is the book sentences dataset. This one is very clean and doesn't need us to do much to it.

In [24]:
import numpy as np

books = load_dataset('SamuelYang/bookcorpus', split='train')

# bad idea to try and load all datapoints in memory
# get article count of 198085 at random instead 
rng = np.random.default_rng(1)
# indexes of articles selected by the RNG
idx = rng.choice(len(books), size=198085, replace=False)

# then turn those into a df
df = books.select(idx).to_pandas()

pd.DataFrame({
    'id': range(len(df)),
    'src': 'books',
    'txt': df['text'].astype(str).to_numpy()
}).to_parquet('../datasets/books.parquet', index=False)